# 単語の頻度を数える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_corpus_statistics/word_freq.ipynb)

NLP クックブック 07-02 ／ コーパスの統計と可視化

テキスト中でどの単語が何回出てくるかを数えて、頻度順に並べたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile

def aozora(url):
    """青空文庫の zip (ルビあり) を取得して本文だけを返す"""
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

text = aozora("https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip")  # 吾輩は猫である
print(len(text), "文字")

321138 文字


In [3]:
from collections import Counter
from fugashi import Tagger

tagger = Tagger()
# 注意: fugashi のノードは次の解析で上書きされるので、必要な属性をその場で取り出して保存する
words = [(w.surface, w.feature.pos1, w.feature.pos2, w.feature.orthBase or w.surface)
         for line in text.split("\n") for w in tagger(line)]
print("延べ語数:", len(words))

freq_all = Counter(surface for surface, *_ in words)
print(freq_all.most_common(15))

# 名詞だけ (原形で集計)
freq_noun = Counter(base for surface, pos1, pos2, base in words if pos1 == "名詞" and pos2 in ("普通名詞", "固有名詞"))
print(freq_noun.most_common(15))

延べ語数: 209274
[('の', 9543), ('。', 7486), ('て', 7417), ('に', 7058), ('、', 6773), ('は', 6501), ('と', 6157), ('を', 6119), ('が', 5395), ('で', 4543), ('た', 3974), ('も', 3238), ('「', 3238), ('」', 3238), ('だ', 2705)]
[('事', 1211), ('もの', 975), ('主人', 934), ('ところ', 386), ('人', 369), ('時', 353), ('迷亭', 343), ('寒月', 286), ('方', 284), ('人間', 279), ('顔', 277), ('先生', 274), ('上', 258), ('猫', 237), ('細君', 212)]


In [4]:
import pandas as pd

df = pd.DataFrame(freq_noun.most_common(), columns=["word", "freq"])
df["ratio"] = df["freq"] / df["freq"].sum()
df["cumulative"] = df["ratio"].cumsum()
df.head(10)

,word,freq,ratio,cumulative
0,事,1211,0.027870,0.027870
1,もの,975,0.022439,0.050310
2,主人,934,0.021495,0.071805
3,ところ,386,0.008884,0.080689
4,人,369,0.008492,0.089181
5,時,353,0.008124,0.097305
6,迷亭,343,0.007894,0.105199
7,寒月,286,0.006582,0.111781
8,方,284,0.006536,0.118317
9,人間,279,0.006421,0.124738


- 何も絞らないと上位は助詞と句読点で埋まる。見たいものに応じて品詞で絞る。
- `[w for line in lines for w in tagger(line)]` のようにノードそのものを溜め込むと、中身が壊れて品詞や原形がでたらめになる (fugashi の落とし穴)。